# Data Acquisition & Select Mortality Extraction

In [1]:
import pandas as pd
import numpy as np

# 1. Load the raw file
file_path = 'Export.csv'
raw_data = pd.read_csv(file_path, header=None)

# 2. Dynamically find the header row (where 'Row\Column' is located)
try:
    header_idx = raw_data[raw_data[0] == "Row\\Column"].index[0]
except IndexError:
    print("Error: Could not find the 'Row\Column' marker. Check the CSV format.")
    exit()

# 3. Reload the data starting from that header
df = pd.read_csv(file_path, skiprows=header_idx)

# 4. Target Issue Age 35
issue_age = 35
term_length = 20

# Filter the first column (Issue Age) for our target
# We take the first match [0] to ensure we get the 'Select' rates, not 'Ultimate'
target_row = df[df.iloc[:, 0].astype(str) == str(issue_age)].iloc[0]

# 5. Extract mortality rates (qx) for durations 1 through 20
# We skip index 0 (the age) and take the next 20 values
qx_values = target_row.iloc[1:term_length + 1].values

# 6. Create the Cleaned DataFrame
cleaned_vbt = pd.DataFrame({
    'Policy_Year': range(1, term_length + 1),
    'Attained_Age': range(issue_age, issue_age + term_length),
    'qx': pd.to_numeric(qx_values)
})

# 7. Save to CSV for Milestone 2
cleaned_vbt.to_csv('cleaned_mortality_35.csv', index=False)

print("--- Extraction Successful ---")
print(cleaned_vbt.head())

<>:12: SyntaxWarning: invalid escape sequence '\C'
<>:12: SyntaxWarning: invalid escape sequence '\C'
C:\Users\AJITH\AppData\Local\Temp\ipykernel_1912\2590749053.py:12: SyntaxWarning: invalid escape sequence '\C'
  print("Error: Could not find the 'Row\Column' marker. Check the CSV format.")


--- Extraction Successful ---
   Policy_Year  Attained_Age       qx
0            1            35  0.00021
1            2            36  0.00029
2            3            37  0.00042
3            4            38  0.00049
4            5            39  0.00057


# Actuarial Foundations – Discounting & Survival Probabilities

In [2]:
import pandas as pd
import numpy as np

# Load your cleaned data
df = pd.read_csv('cleaned_mortality_35.csv')

# 1. Financial Assumptions
i = 0.05
v = 1 / (1 + i)

# 2. Calculate p_x
df['px'] = 1 - df['qx']

# 3. Calculate tpx (Probability of being alive at the START of the year)
df['tpx'] = df['px'].shift(1, fill_value=1.0).cumprod()

# 4. Calculate Discount Factors
# Premiums: Paid at start (Time 0, 1, 2...) -> v^0, v^1, v^2...
df['v_premium'] = v ** (df['Policy_Year'] - 1)

# Benefits: NOW paid at end of year (Time 1, 2, 3...) -> v^1, v^2, v^3...
# CHANGED: Removed the -0.5 adjustment
df['v_benefit'] = v ** df['Policy_Year'] 

# 5. Expected Present Values for $1
# EPV Benefit = Discount(at end of yr) * Prob(Alive at start) * Prob(Die during yr)
df['EPV_Benefit_Base'] = df['v_benefit'] * df['tpx'] * df['qx']
df['EPV_Premium_Base'] = df['v_premium'] * df['tpx']

# Save the progress
df.to_csv('actuarial_values_35_EOY.csv', index=False)

print("--- Actuarial Columns Added (End-of-Year Assumption) ---")
print(df[['Policy_Year', 'tpx', 'qx', 'v_benefit', 'EPV_Benefit_Base']].head())

--- Actuarial Columns Added (End-of-Year Assumption) ---
   Policy_Year       tpx       qx  v_benefit  EPV_Benefit_Base
0            1  1.000000  0.00021   0.952381          0.000200
1            2  0.999790  0.00029   0.907029          0.000263
2            3  0.999500  0.00042   0.863838          0.000363
3            4  0.999080  0.00049   0.822702          0.000403
4            5  0.998591  0.00057   0.783526          0.000446


# Policy Pricing – Applying the Equivalence Principle

In [3]:
# 1. Define Face Amount
face_amount = 100000

# 2. Sum the EPV bases we calculated in the previous step
total_epv_benefits = df['EPV_Benefit_Base'].sum() * face_amount
total_epv_premiums_factor = df['EPV_Premium_Base'].sum()  # This is the Annuity Factor

# 3. Calculate the Net Level Premium
net_premium = total_epv_benefits / total_epv_premiums_factor

print(f"--- Final Pricing Result ---")
print(f"Total APV of Benefits: ${total_epv_benefits:,.2f}")
print(f"Annuity Factor: {total_epv_premiums_factor:.4f}")
print(f"Annual Net Level Premium: ${net_premium:,.2f}")

# Optional: Save a summary for your report
summary = pd.DataFrame({
    'Metric': ['Face Amount', 'Interest Rate', 'APV Benefits', 'Annuity Factor', 'Annual Net Premium'],
    'Value': [face_amount, i, total_epv_benefits, total_epv_premiums_factor, net_premium]
})
summary.to_csv('pricing_summary.csv', index=False)

--- Final Pricing Result ---
Total APV of Benefits: $1,344.36
Annuity Factor: 13.0076
Annual Net Level Premium: $103.35


# Valuation – Prospective Terminal Reserve Calculation

In [4]:
# 1. Calculate Prospective Reserves for each year
reserves = []
for t in range(1, term_length + 1):
    # Slice the dataframe to look at only FUTURE years (from t to the end)
    future_df = df.iloc[t:].copy()
    
    if future_df.empty:
        reserves.append(0.0) # Reserve at the end of the term is always 0
    else:
        # Re-calculate survival probabilities starting from the current year t
        # Note: df.iloc[t]['tpx'] is survival to the start of year t+1 (which is end of year t)
        tpx_at_t = df.iloc[t]['tpx']
        
        future_benefit_epv = 0
        future_premium_epv = 0
        
        for k in range(len(future_df)):
            # Probability of surviving k more years given alive at time t
            k_px_at_t = future_df.iloc[k]['tpx'] / tpx_at_t
            
            # --- DISCOUNTING FOR END-OF-YEAR ---
            v_b = v ** (k + 1)  # Benefit is 1, 2, 3... years away from time t
            v_p = v ** (k)      # Next premium is due IMMEDIATELY at time t (dist = 0)
            
            future_benefit_epv += face_amount * v_b * k_px_at_t * future_df.iloc[k]['qx']
            future_premium_epv += v_p * k_px_at_t
            
        # The Reserve Formula
        reserve_t = future_benefit_epv - (net_premium * future_premium_epv)
        reserves.append(reserve_t)

# 2. Add to DataFrame
df['Terminal_Reserve'] = reserves

# 3. Save the Master File for Excel
df.to_csv('master_actuarial_model.csv', index=False)

print("--- Python Phase Complete ---")
print(f"Net Premium: ${net_premium:.2f}")
print(df[['Policy_Year', 'Attained_Age', 'qx', 'Terminal_Reserve']].head())

--- Python Phase Complete ---
Net Premium: $103.35
   Policy_Year  Attained_Age       qx  Terminal_Reserve
0            1            35  0.00021         87.537661
1            2            36  0.00029        171.483552
2            3            37  0.00042        246.680614
3            4            38  0.00049        318.690081
4            5            39  0.00057        386.364091


# Sensitivity Analysis

In [5]:
import pandas as pd

# 1. Encapsulate your pricing logic into a reusable function
def calculate_premium_for_rate(i, base_df, face_amount=100000):
    # Create a copy so we don't accidentally overwrite the original data
    df_temp = base_df.copy()
    
    # Financial Assumption
    v = 1 / (1 + i)
    
    # Calculate probabilities (re-applying your Cell 2 logic)
    df_temp['px'] = 1 - df_temp['qx']
    df_temp['tpx'] = df_temp['px'].shift(1, fill_value=1.0).cumprod()
        
    # Vectorized Discounting (End-of-Year assumption)
    df_temp['v_premium'] = v ** (df_temp['Policy_Year'] - 1)
    df_temp['v_benefit'] = v ** df_temp['Policy_Year'] 
    
    # Expected Present Values
    df_temp['EPV_Benefit_Base'] = df_temp['v_benefit'] * df_temp['tpx'] * df_temp['qx']
    df_temp['EPV_Premium_Base'] = df_temp['v_premium'] * df_temp['tpx']
    
    # Equivalence Principle
    total_epv_benefits = df_temp['EPV_Benefit_Base'].sum() * face_amount
    annuity_factor = df_temp['EPV_Premium_Base'].sum()
    
    net_premium = total_epv_benefits / annuity_factor
    
    return net_premium

# 2. Define the Market Risk Scenarios
rates_to_test = [0.02, 0.03, 0.04, 0.05, 0.06]
face_amt = 100000

# Load the base mortality data you cleaned earlier
clean_vbt = pd.read_csv('cleaned_mortality_35.csv')

# 3. Run the analysis
sensitivity_results = []

for rate in rates_to_test:
    # Call our function for each rate
    prem = calculate_premium_for_rate(rate, clean_vbt, face_amt)
    
    # Append the results to a list of dictionaries
    sensitivity_results.append({
        'Interest Rate': f"{rate*100}%",
        'Net Level Premium': round(prem, 2)
    })

# 4. Create a clean DataFrame and export for Excel
df_sensitivity = pd.DataFrame(sensitivity_results)
df_sensitivity.to_csv('premium_sensitivity.csv', index=False)

print("--- Sensitivity Analysis Complete ---")
print(df_sensitivity.to_string(index=False))

--- Sensitivity Analysis Complete ---
Interest Rate  Net Level Premium
         2.0%             118.81
         3.0%             113.42
         4.0%             108.27
         5.0%             103.35
         6.0%              98.67
